# Prism Alpha — nanoGPT Spectral Initialization

Two modes for accelerating GPT training from scratch:

**Sprint** — reach baseline quality in 4.8x fewer steps

**Marathon** — achieve better quality than baseline, with zero overfitting

Both use the same technique: extract the spectral fingerprint from a trained
model, inject it into a fresh model's initialization, and continuously
modulate weights toward the spectral target during training.

---
*Created by [Sean McDonald](https://x.com/seanmcdonaldxyz) · [github.com/realityinspector/nanogpt-prism](https://github.com/realityinspector/nanogpt-prism) · Apache 2.0*

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 1: Setup (run once)
# ═══════════════════════════════════════════════════════
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets

import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n  GPU: {gpu} ({vram:.0f} GB)')

# Prepare Shakespeare dataset
!python data/shakespeare_char/prepare.py 2>/dev/null
print('  Dataset: Shakespeare (char-level)')
print('  Ready.\n')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 2: Train teacher + extract spectral fingerprint
# (run once — cached for subsequent runs)
# ═══════════════════════════════════════════════════════
import os, subprocess

if os.path.exists('.prism_cache/shakespeare/spectra.json'):
    print('  Spectral fingerprint already cached.\n')
else:
    print('  Training teacher model to convergence (2K steps)...')
    subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        '--max_iters=2000', '--eval_interval=2000', '--eval_iters=50',
        '--log_interval=500', '--out_dir=out-teacher',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False',
    ], capture_output=True, timeout=600)
    
    print('  Extracting spectral fingerprint (32 DCT coefficients + directions)...')
    result = subprocess.run([
        'python', 'prism_extract.py',
        '--ckpt', 'out-teacher/ckpt.pt',
        '--out', '.prism_cache/shakespeare',
    ], capture_output=True, text=True, timeout=120)
    print(result.stdout)
    print('  Done.\n')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 3: Run all three — Baseline, Sprint, Marathon
# ═══════════════════════════════════════════════════════
import subprocess, time, re, json, os

STEPS = 5000

PRISM = [
    '--prism_init=True', '--prism_align=0.75',
    '--prism_spectra=.prism_cache/shakespeare/spectra.json',
    '--prism_directions=.prism_cache/shakespeare/directions.pt',
]

runs = [
    ('Baseline', ['--prism_init=False']),
    ('Prism Sprint', PRISM + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999',
    ]),
    ('Prism Marathon', PRISM + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999',
    ]),
]

all_results = {}

for display_name, extra in runs:
    tag = display_name.lower().replace(' ', '_')
    log_path = f'/content/alpha_{tag}.txt'
    
    print(f'\n  {"═"*55}')
    print(f'  {display_name}')
    print(f'  {"═"*55}')
    
    t0 = time.time()
    result = subprocess.run(
        ['python', 'train.py', 'config/train_shakespeare_char.py',
         f'--max_iters={STEPS}', '--eval_interval=100',
         '--eval_iters=50', '--log_interval=250',
         f'--out_dir=out-alpha-{tag}',
         '--wandb_log=False', '--compile=False'] + extra,
        capture_output=True, text=True, timeout=1200
    )
    wall = time.time() - t0
    
    with open(log_path, 'w') as f:
        f.write(result.stdout)
    
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[-300:]}')
        continue
    
    # Parse val losses
    val = {}
    for line in result.stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val[int(m.group(1))] = float(m.group(3))
    
    best_v = min(val.values()) if val else 999
    best_s = min(val, key=val.get) if val else 0
    at_5k = val.get(STEPS, val.get(max(val.keys()), 0)) if val else 0
    
    all_results[display_name] = {
        'val_losses': val, 'best_val': best_v,
        'best_step': best_s, 'at_5000': at_5k, 'wall_time': wall,
    }
    print(f'  Best val loss: {best_v:.4f} at step {best_s}')
    print(f'  Val @ step 5000: {at_5k:.4f}')
    print(f'  Wall time: {wall:.0f}s')

# Save
with open('/content/alpha_results.json', 'w') as f:
    save = {}
    for k, v in all_results.items():
        save[k] = {kk: vv for kk, vv in v.items() if kk != 'val_losses'}
        save[k]['val_losses'] = {str(s): l for s, l in v['val_losses'].items()}
    json.dump(save, f, indent=2)
print('\n  Results saved to /content/alpha_results.json')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 4: Results
# ═══════════════════════════════════════════════════════
import json

data = json.load(open('/content/alpha_results.json'))
for k in data:
    data[k]['val_losses'] = {int(s): v for s, v in data[k]['val_losses'].items()}

b = data['Baseline']
s = data['Prism Sprint']
m = data['Prism Marathon']

target = b['best_val']
target_step = b['best_step']

def steps_to_target(d, t):
    for step in sorted(d['val_losses'].keys()):
        if d['val_losses'][step] <= t:
            return step
    return None

s_hit = steps_to_target(s, target)
m_hit = steps_to_target(m, target)

print()
print('  ┌─────────────────────────────────────────────────────────┐')
print('  │              PRISM ALPHA — RESULTS                      │')
print('  ├─────────────────────────────────────────────────────────┤')
print(f'  │  Baseline best: {target:.4f} at step {target_step:>5d}                  │')
print('  ├──────────────┬──────────┬──────────┬────────┬──────────┤')
print('  │ Mode         │ Steps to │ Speedup  │ Best   │ @5000    │')
print('  │              │ baseline │          │ val    │ val      │')
print('  ├──────────────┼──────────┼──────────┼────────┼──────────┤')
print(f'  │ Baseline     │ {target_step:>8d} │    1.0x  │{target:>7.4f} │{b["at_5000"]:>9.4f} │')
s_spd = f'{target_step/s_hit:.1f}x' if s_hit else '—'
print(f'  │ Sprint       │ {str(s_hit or "never"):>8s} │ {s_spd:>7s}  │{s["best_val"]:>7.4f} │{s["at_5000"]:>9.4f} │')
m_spd = f'{target_step/m_hit:.1f}x' if m_hit else '—'
print(f'  │ Marathon     │ {str(m_hit or "never"):>8s} │ {m_spd:>7s}  │{m["best_val"]:>7.4f} │{m["at_5000"]:>9.4f} │')
print('  └──────────────┴──────────┴──────────┴────────┴──────────┘')

print(f'\n  Sprint: {s_spd} faster to reach baseline quality. Stop at step ~600.')
print(f'  Marathon: best val {m["best_val"]:.4f} — lower than baseline ever achieves.')
print(f'  Marathon @5000: {m["at_5000"]:.4f} vs baseline @5000: {b["at_5000"]:.4f} (baseline overfit)')

# Convergence curve
print(f'\n  {"Step":>6s}  {"Baseline":>10s}  {"Sprint":>10s}  {"Marathon":>10s}')
print(f'  {"-"*42}')
for step in sorted(set(b['val_losses'].keys()) & set(s['val_losses'].keys()) & set(m['val_losses'].keys())):
    if step % 500 == 0 or step <= 500 and step % 100 == 0:
        bv = b['val_losses'].get(step, 0)
        sv = s['val_losses'].get(step, 0)
        mv = m['val_losses'].get(step, 0)
        best = min(bv, sv, mv)
        bm = ' *' if abs(bv-best)<0.002 else '  '
        sm = ' *' if abs(sv-best)<0.002 else '  '
        mm = ' *' if abs(mv-best)<0.002 else '  '
        print(f'  {step:>6d}  {bv:>8.4f}{bm}  {sv:>8.4f}{sm}  {mv:>8.4f}{mm}')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 5: Download results
# ═══════════════════════════════════════════════════════
from google.colab import files
files.download('/content/alpha_results.json')
for tag in ['baseline', 'prism_sprint', 'prism_marathon']:
    try: files.download(f'/content/alpha_{tag}.txt')
    except: pass